In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.functions import left

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

flag_run_task = dbutils.jobs.taskValues.get(taskKey = "00_FILE_INGEST", key = "run_tsk")

if flag_run_task:
    entry_file_schema = StructType([
        StructField("cliente", StringType(), True),
        StructField("cnpj", StringType(), True),
        StructField("descricao_do_projeto", StringType(), True),
        StructField("uf", StringType(), True),
        StructField("municipio", StringType(), True),
        StructField("municipio_codigo", StringType(), True),
        StructField("numero_do_contrato", StringType(), True),
        StructField("data_da_contratacao", StringType(), True),
        StructField("valor_contratado_reais", StringType(), True),
        StructField("valor_desenbolsado_reais", StringType(), True),
        StructField("origem_recurso_desembolsos", StringType(), True),
        StructField("custo_financeiro", StringType(), True),
        StructField("juros", StringType(), True),
        StructField("prazo_carencia_meses", StringType(), True),
        StructField("prazo_amortizacao_meses", StringType(), True),
        StructField("modalidade_apoio", StringType(), True),
        StructField("forma_de_apoio", StringType(), True),
        StructField("produto", StringType(), True),
        StructField("instrumento_financeiro", StringType(), True),
        StructField("inovacao", StringType(), True),
        StructField("area_operacional", StringType(), True),
        StructField("setor_cnae", StringType(), True),
        StructField("subsetor_cnae_agrupado", StringType(), True),
        StructField("subsetor_cnae_codigo", StringType(), True),
        StructField("subsetor_cnae_nome", StringType(), True),
        StructField("setor_bndes", StringType(), True),
        StructField("subsetor_bndes", StringType(), True),
        StructField("porte_do_cliente", StringType(), True),
        StructField("natureza_do_cliente", StringType(), True),
        StructField("instituicao_financeira_credenciada", StringType(), True),
        StructField("cnpj_instituicao_financeira_credenciada", StringType(), True),
        StructField("tipo_de_garantia", StringType(), True),
        StructField("tipo_de_excepcionalidade", StringType(), True),
        StructField("situacao_do_contrato", StringType(), True),
    ])

    entry_file_raw = dbutils.jobs.taskValues.get(taskKey = "00_FILE_INGEST", key = "entry_file_raw_tsk")

    file_data_ref = dbutils.jobs.taskValues.get(taskKey = "00_FILE_INGEST", key = "file_data_ref_tsk")

    entry_file = spark.read\
        .format("csv")\
        .option("header", "true")\
        .option("sep", ";")\
        .schema(entry_file_schema)\
        .option("encoding", "iso-8859-1")\
        .load(entry_file_raw)

    df_entry_file = entry_file\
        .withColumn("cnpj", 
            lpad(concat(substring(lpad(regexp_replace(col("cnpj"), "[/.,-]",""), 15, "0"),4,9),lit('***')),15,"*")
       )\
        .withColumn("numero_do_contrato", 
           lpad(regexp_replace(col("numero_do_contrato"), "[/.,-]",""), 9, "0")
        )\
       .withColumn("cnpj_instituicao_financeira_credenciada", 
           lpad(regexp_replace(col("cnpj_instituicao_financeira_credenciada"), "[/.,-]",""), 15, "0")
        )\
        .withColumn("data_da_contratacao", 
            date_format(to_date(col("data_da_contratacao"), "dd/MM/yyyy"), "yyyy-MM-dd")
        )\
        .withColumn("valor_contratado_reais", 
            regexp_replace(col("valor_contratado_reais"), ",",".").cast("double")
        )\
        .withColumn("valor_desenbolsado_reais", 
            regexp_replace(col("valor_desenbolsado_reais"), ",",".").cast("double")
        )\
        .withColumn("juros", 
            regexp_replace(col("juros"), ",",".").cast("double")
        )\
        .withColumn("prazo_carencia_meses", 
            regexp_replace(col("prazo_carencia_meses"), ",",".").cast("integer")
        )\
        .withColumn("prazo_amortizacao_meses", 
            regexp_replace(col("prazo_amortizacao_meses"), ",",".").cast("integer")
        )\
        .withColumn("cnpj_instituicao_financeira_credenciada", 
            lpad(concat(substring(lpad(regexp_replace(col("cnpj_instituicao_financeira_credenciada"), "[/.,-]",""), 15, "0"),4,8),lit('****')),15,"*")
        )\
        .withColumn("data_ref", 
            lit(file_data_ref)
        )
        
    try:
        dt_ref_bronze = spark.table("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto").agg(max(col("data_ref"))).first()[0]
        dt_ref_update = df_entry_file.agg(max(col("data_ref"))).first()[0]
        if dt_ref_update > dt_ref_update:
            try:
                print("appendando base atualizada")
                df_entry_file.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto")
            except:
                print("erro ao appendar")
        elif dt_ref_update == dt_ref_bronze:
            print("erro: base com referencia igual a ultima atualização")
        else:
            print("erro: base com referencia anterior a ultima atualização")
    except:
        print("criando base")
        df_entry_file.write.mode("append").partitionBy("data_ref").format("delta").saveAsTable("data_warehouse.bndes.BRONZE_LAYER_bndes_oper_finan_n_auto")

else: print(flag_run_task)